# Charbonneau (2010) — Dynamo Models of the Solar Cycle: Implementation
# Charbonneau (2010) — 태양 주기의 다이나모 모델: 구현

**English.** This notebook implements simple kinematic dynamo models from Charbonneau (2010, LRSP 7, 3): (1) the helioseismic differential rotation profile, (2) a 1-D Parker dynamo wave illustrating the $\alpha\Omega$ dispersion and equatorward propagation, (3) a low-order coupled $\alpha\Omega$ ODE model demonstrating the dynamo number criticality, (4) a Babcock–Leighton flux-transport dynamo in latitude generating a butterfly diagram, (5) the time-delay iterative B–L map showing intermittency, and (6) a stochastic B–L cycle simulation producing Maunder-like grand minima.

**한국어.** 이 노트북은 Charbonneau (2010)의 단순 운동학적 다이나모 모델을 구현합니다: (1) 일진동학적 미분 회전 프로파일, (2) Parker 다이나모 파동의 $\alpha\Omega$ 분산 및 적도방향 전파를 보여주는 1차원 모델, (3) 다이나모 수 임계성을 보여주는 저차원 결합 $\alpha\Omega$ ODE 모델, (4) 위도 방향 Babcock–Leighton 플럭스 수송 다이나모로 버터플라이 다이어그램 생성, (5) 간헐성을 보여주는 시간지연 B–L 반복 사상, (6) Maunder형 grand minima를 만드는 확률적 B–L 주기 시뮬레이션.

In [ ]:
"""Imports and global plotting style.

Standard scientific Python stack. We use np.trapezoid (renamed from
np.trapz in NumPy 2.x) for any integrals.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

rng = np.random.default_rng(42)
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("NumPy version:", np.__version__)

## Part 1 — Helioseismic Differential Rotation $\Omega(r,\theta)$ / 일진동학적 미분 회전

**English.** The single most influential observational input to modern solar dynamos is the helioseismic rotation profile. We implement the canonical fit (Charbonneau §4.1) used in essentially every flux-transport model:
$$\Omega(r,\theta) = \Omega_C + \tfrac{1}{2}\!\left[1 + \mathrm{erf}\!\left(\tfrac{r-r_c}{w}\right)\right]\!\left[\Omega_S(\theta) - \Omega_C\right],$$
with $\Omega_S(\theta) = \Omega_{\rm eq}(1 - a_2\cos^2\theta - a_4\cos^4\theta)$.

**한국어.** 현대 태양 다이나모에서 가장 영향력 있는 관측 입력은 일진동학적 회전 프로파일입니다. 거의 모든 플럭스 수송 모델이 사용하는 표준 적합식을 구현합니다.

In [ ]:
from scipy.special import erf


def helioseismic_omega(r: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """Compute helioseismic angular velocity Omega(r, theta) in nHz.

    Args:
        r: Fractional radius r/R_sun, shape (Nr,).
        theta: Colatitude in radians, shape (Ntheta,).

    Returns:
        Omega in nHz, shape (Nr, Ntheta).
    """
    omega_core = 432.0  # nHz, rigid radiative interior
    omega_eq = 460.0    # nHz, equatorial surface rate
    a2, a4 = 0.17, 0.08
    rc, w = 0.7, 0.025

    R, T = np.meshgrid(r, theta, indexing="ij")
    omega_surface = omega_eq * (1.0 - a2 * np.cos(T) ** 2 - a4 * np.cos(T) ** 4)
    transition = 0.5 * (1.0 + erf((R - rc) / w))
    return omega_core + transition * (omega_surface - omega_core)


r_grid = np.linspace(0.55, 1.0, 200)
theta_grid = np.linspace(1e-3, np.pi - 1e-3, 200)
Omega = helioseismic_omega(r_grid, theta_grid)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
im = ax[0].contourf(
    np.rad2deg(theta_grid) - 90.0, r_grid, Omega, levels=20, cmap="viridis"
)
plt.colorbar(im, ax=ax[0], label="Omega [nHz]")
ax[0].axhline(0.7, color="r", lw=1.2, ls="--", label="tachocline")
ax[0].set_xlabel("latitude [deg]")
ax[0].set_ylabel("r / R_sun")
ax[0].set_title("Helioseismic Omega(r, theta) / 일진동학 Ω(r,θ)")
ax[0].legend(loc="lower right")

for r0, label in [(0.7, "tachocline"), (0.85, "mid CZ"), (1.0, "surface")]:
    idx = np.argmin(np.abs(r_grid - r0))
    ax[1].plot(np.rad2deg(theta_grid) - 90.0, Omega[idx], label=f"r={r0} {label}")
ax[1].set_xlabel("latitude [deg]")
ax[1].set_ylabel("Omega [nHz]")
ax[1].set_title("Latitudinal cuts / 위도 단면")
ax[1].legend()
plt.tight_layout()
plt.show()

## Part 2 — Parker Dynamo Wave (1-D Cartesian) / Parker 다이나모 파동 (1차원)

**English.** In the local Cartesian limit with uniform shear $G = \partial\Omega/\partial r$ and constant $\alpha$, the linearized $\alpha\Omega$ system supports plane-wave solutions $A,B \propto \exp(iky - i\omega t)$ with dispersion $(i\omega + \eta_T k^2)^2 = i\alpha G k$. Real part of $\omega$ → oscillation, imaginary part → growth. The phase velocity sign gives the Parker–Yoshimura rule.

**한국어.** 균일 전단 $G$와 상수 $\alpha$의 국소 직교 한계에서 선형 $\alpha\Omega$ 시스템은 평면파 해를 지지하며, 분산 관계 $(i\omega + \eta_T k^2)^2 = i\alpha G k$를 갖습니다. $\omega$의 실수부는 진동, 허수부는 성장률입니다.

In [ ]:
def parker_dispersion(
    k: np.ndarray, alpha: float, G: float, eta_T: float
) -> tuple[np.ndarray, np.ndarray]:
    """Compute Parker dynamo-wave frequency and growth rate.

    Solves (i*omega + eta_T*k^2)^2 = i * alpha * G * k for omega.

    Args:
        k: Wavenumber array.
        alpha: Alpha-effect coefficient.
        G: Shear dOmega/dr.
        eta_T: Turbulent diffusivity.

    Returns:
        Tuple (omega_real, omega_imag) of cycle frequency and growth rate.
    """
    # Right-hand side of the dispersion relation.
    rhs = 1j * alpha * G * k
    sqrt_rhs = np.sqrt(rhs)  # principal branch
    # Two roots: i*omega + eta_T*k^2 = +/- sqrt(rhs).
    omega_p = (sqrt_rhs - eta_T * k ** 2) / 1j
    omega_m = (-sqrt_rhs - eta_T * k ** 2) / 1j
    # Pick the physically growing branch.
    growing = np.where(omega_p.imag > omega_m.imag, omega_p, omega_m)
    return growing.real, growing.imag


k_arr = np.linspace(0.1, 5.0, 200)
alpha0, G0, eta_T0 = 1.0, -10.0, 0.1  # arbitrary units; alpha*G < 0 for equatorward
omega_re, omega_im = parker_dispersion(k_arr, alpha0, G0, eta_T0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(k_arr, omega_re, label="Re(omega) (cycle freq.)")
ax[0].plot(k_arr, omega_im, label="Im(omega) (growth rate)")
ax[0].axhline(0, color="k", lw=0.5)
ax[0].set_xlabel("k")
ax[0].set_ylabel("omega")
ax[0].set_title("Parker dispersion / Parker 분산")
ax[0].legend()

# Reconstruct the wave in the (y, t) plane to visualize migration.
k_pick = 1.5
wr, wi = parker_dispersion(np.array([k_pick]), alpha0, G0, eta_T0)
y = np.linspace(0, 2 * np.pi, 200)
t = np.linspace(0, 4 * np.pi, 200)
Y, T = np.meshgrid(y, t)
wave = np.real(np.exp(1j * (k_pick * Y - wr[0] * T))) * np.exp(wi[0] * T)
im = ax[1].pcolormesh(y, t, wave, cmap="RdBu_r", shading="auto")
ax[1].set_xlabel("y (latitude proxy)")
ax[1].set_ylabel("t")
ax[1].set_title("Migrating dynamo wave / 이동하는 다이나모 파")
plt.colorbar(im, ax=ax[1])
plt.tight_layout()
plt.show()

print(f"For k={k_pick}: cycle period T = {2*np.pi/wr[0]:.3f}, growth rate = {wi[0]:.3f}")

## Part 3 — Coupled $\alpha\Omega$ ODE Model: Dynamo Number Criticality / 결합 $\alpha\Omega$ ODE: 다이나모 수 임계성

**English.** A minimal two-mode reduction of the mean-field $\alpha\Omega$ system replaces the spatial PDE with two coupled ODEs for poloidal $A(t)$ and toroidal $B(t)$ amplitudes:
$$\dot A = -A + C_\alpha\,B, \qquad \dot B = -B + C_\Omega\,A.$$
Eigenvalues $\lambda = -1 \pm \sqrt{C_\alpha C_\Omega} = -1 \pm \sqrt{D}$ → growing for $D > 1$ (the criticality condition). Adding a complex $C_\alpha$ produces oscillation. We sweep $D$ to demonstrate the bifurcation.

**한국어.** 평균장 $\alpha\Omega$ 시스템의 최소 2모드 환원: 공간 PDE를 폴로이달 $A(t)$, 토로이달 $B(t)$ 진폭에 대한 두 결합 ODE로 대체합니다. 고유값 $\lambda = -1 \pm \sqrt{D}$ → $D > 1$에서 성장(임계 조건). 복소 $C_\alpha$를 사용하면 진동도 추가됩니다.

In [ ]:
def alpha_omega_rhs(t: float, y: np.ndarray, c_alpha: complex, c_omega: float) -> np.ndarray:
    """Right-hand side of the two-mode alpha-Omega system.

    State vector y = (Re A, Im A, Re B, Im B). Complex c_alpha allows oscillation.
    """
    A = y[0] + 1j * y[1]
    B = y[2] + 1j * y[3]
    dA = -A + c_alpha * B
    dB = -B + c_omega * A
    return np.array([dA.real, dA.imag, dB.real, dB.imag])


fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, D, label in zip(axes, [0.5, 1.5, 4.0], ["D=0.5 (decay)", "D=1.5 (slow growth)", "D=4 (oscillation+growth)"]):
    c_alpha = np.sqrt(D) * np.exp(1j * np.pi / 4)  # introduce phase for oscillation
    c_omega = np.sqrt(D) * np.exp(-1j * np.pi / 4)
    # Force real c_omega to keep ODE shape, set imaginary part via c_alpha alone.
    sol = solve_ivp(
        alpha_omega_rhs,
        (0, 30),
        [1.0, 0.0, 0.0, 0.0],
        args=(complex(c_alpha), float(np.real(c_omega))),
        t_eval=np.linspace(0, 30, 600),
        method="RK45",
    )
    A_amp = np.hypot(sol.y[0], sol.y[1])
    B_amp = np.hypot(sol.y[2], sol.y[3])
    ax.plot(sol.t, A_amp, label="|A| poloidal")
    ax.plot(sol.t, B_amp, label="|B| toroidal")
    ax.set_yscale("log")
    ax.set_xlabel("time")
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.show()

print("Criticality demonstrated: D < 1 decays, D > 1 grows; phase in c_alpha gives oscillation.")

## Part 4 — Babcock–Leighton Flux-Transport Dynamo: Butterfly Diagram / B–L 플럭스 수송 다이나모: 버터플라이

**English.** A 1-D-in-latitude flux-transport B–L dynamo. We track the toroidal field $B(\theta, t)$ at the tachocline. The poloidal field is generated at the surface by the B–L source, advected to high latitudes by the surface poleward flow, then to the tachocline by the meridional return flow. Equatorward advection of $B$ at the tachocline creates the equatorward butterfly.

Equation (toroidal at tachocline, in latitude $\lambda = 90° - \theta$):
$$\frac{\partial B}{\partial t} = -u_\lambda \frac{\partial B}{\partial \lambda} + S_\Omega\,P(\lambda, t-\tau) + \eta_T\frac{\partial^2 B}{\partial \lambda^2} - \frac{B}{\tau_d},$$
where $u_\lambda < 0$ in N hemisphere (equatorward at the tachocline), $S_\Omega$ is the $\Omega$-shear source acting on a delayed poloidal field $P(\lambda, t-\tau)$, and $P$ is sourced near the surface by $\sin(2\lambda)$ from the B–L mechanism applied to $|B|$.

**한국어.** 위도 1차원 플럭스 수송 B–L 다이나모. tachocline의 토로이달 $B(\theta, t)$를 추적합니다. 폴로이달은 표면 B–L 원천에서 생성되어 극으로 이류, 자오선 반환류로 tachocline에 도달, 적도방향 이류로 적도방향 버터플라이를 만듭니다.

In [ ]:
def run_bl_butterfly(
    n_lat: int = 90,
    n_t: int = 4000,
    t_max: float = 80.0,
    u0: float = 1.5,
    s_omega: float = 6.0,
    tau_decay: float = 12.0,
    eta: float = 0.05,
    delay_yr: float = 1.5,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run a 1-D Babcock-Leighton flux-transport dynamo and return B, P, t.

    Args:
        n_lat: Number of latitude grid points (-90 to +90 deg).
        n_t: Number of timesteps.
        t_max: Total simulation time in years.
        u0: Equatorward flow speed (deg/yr) at the tachocline.
        s_omega: Toroidal source strength from poloidal shearing.
        tau_decay: Toroidal decay time (years), models flux loss.
        eta: Latitudinal diffusivity (deg^2/yr).
        delay_yr: Delay between surface poloidal source and tachocline toroidal effect.

    Returns:
        Tuple (B, P, t) of toroidal-time grid, poloidal-time grid, and time array.
    """
    lat = np.linspace(-90.0, 90.0, n_lat)
    dlat = lat[1] - lat[0]
    t = np.linspace(0.0, t_max, n_t)
    dt = t[1] - t[0]
    delay_steps = max(1, int(round(delay_yr / dt)))

    B = np.zeros((n_t, n_lat))  # toroidal at tachocline
    P = np.zeros((n_t, n_lat))  # poloidal at surface (latitude-time)
    # Initial seed: weak antisymmetric toroidal field.
    B[0] = 0.01 * np.sin(np.deg2rad(2 * lat))

    for n in range(n_t - 1):
        # Equatorward advection at tachocline: u_lambda = -sign(lat)*u0.
        u_lat = -np.sign(lat) * u0
        dBdlat = np.gradient(B[n], dlat)
        d2Bdlat2 = np.gradient(dBdlat, dlat)
        # Delayed poloidal -> toroidal coupling via Omega shear.
        n_delay = max(0, n - delay_steps)
        # Toroidal source localized in mid-latitudes (where shear is strong).
        toroidal_source = (
            s_omega * P[n_delay] * np.cos(np.deg2rad(lat)) ** 2
        )
        B[n + 1] = (
            B[n]
            + dt * (-u_lat * dBdlat + toroidal_source
                    + eta * d2Bdlat2 - B[n] / tau_decay)
        )
        # Babcock-Leighton: poloidal source at surface from |B| at tachocline,
        # antisymmetric across equator (Joy-law style), saturating in B.
        bl_source = (
            np.sin(np.deg2rad(2 * lat))
            * B[n]
            / (1.0 + (B[n] / 1.5) ** 2)
        )
        # Poloidal evolves with surface poleward advection and decay.
        u_lat_surf = np.sign(lat) * 0.5 * u0  # poleward at surface
        dPdlat = np.gradient(P[n], dlat)
        P[n + 1] = (
            P[n]
            + dt * (-u_lat_surf * dPdlat + 0.6 * bl_source
                    + eta * np.gradient(dPdlat, dlat) - P[n] / (2 * tau_decay))
        )
    return B, P, t


B_field, P_field, t_arr = run_bl_butterfly()
lat_arr = np.linspace(-90.0, 90.0, B_field.shape[1])

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
vmax = np.percentile(np.abs(B_field), 99)
im0 = ax[0].pcolormesh(
    t_arr, lat_arr, B_field.T, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto"
)
ax[0].set_ylabel("latitude [deg]")
ax[0].set_title("Toroidal field B(lat, t) — Butterfly diagram / 토로이달 자기장 — 버터플라이")
ax[0].set_ylim(-60, 60)
plt.colorbar(im0, ax=ax[0], label="B (a.u.)")

vmax_p = np.percentile(np.abs(P_field), 99)
im1 = ax[1].pcolormesh(
    t_arr, lat_arr, P_field.T, cmap="PuOr_r", vmin=-vmax_p, vmax=vmax_p, shading="auto"
)
ax[1].set_xlabel("time [yr]")
ax[1].set_ylabel("latitude [deg]")
ax[1].set_title("Poloidal field P(lat, t) — polar drift / 폴로이달 — 극 이동")
plt.colorbar(im1, ax=ax[1], label="P (a.u.)")
plt.tight_layout()
plt.show()

## Part 5 — Time-Delay B–L Iterative Map: Period Doubling and Chaos / 시간지연 B–L 반복 사상: 주기배가와 혼돈

**English.** Charbonneau §5.4 reduces the time-delay B–L dynamo to an iterated map. With $T_n$ = toroidal amplitude at cycle $n$ and a saturating B–L source, the map is
$$T_{n+1} = a\,T_n / [1 + (T_n / T_*)^2].$$
We sweep $a$ and plot the bifurcation diagram, showing fixed points → period doubling → chaos — a *deterministic* origin for cycle-amplitude irregularity.

**한국어.** Charbonneau §5.4는 시간지연 B–L 다이나모를 반복 사상으로 환원합니다. $a$ 값을 변화시키며 분기 다이어그램을 그리면 고정점 → 주기배가 → 혼돈으로 이어집니다. 즉, 주기 진폭 불규칙성의 *결정론적* 기원입니다.

In [ ]:
def bl_map_iterate(a: float, T_star: float = 1.0, n_iter: int = 600,
                   n_keep: int = 200, T0: float = 0.5) -> np.ndarray:
    """Iterate the B-L saturating map T_{n+1} = a*T_n / (1 + (T_n/T_star)^2).

    Args:
        a: Linear growth coefficient (the bifurcation parameter).
        T_star: Saturation amplitude.
        n_iter: Total iterations.
        n_keep: Number of trailing iterations to return for the attractor.
        T0: Initial amplitude.

    Returns:
        Array of trailing T values forming the attractor.
    """
    T = T0
    series = np.empty(n_keep)
    for n in range(n_iter):
        T = a * T / (1.0 + (T / T_star) ** 2)
        if n >= n_iter - n_keep:
            series[n - (n_iter - n_keep)] = T
    return series


# Note: the classic logistic-style bifurcation needs a map that can fold; the
# saturating map alone has a unique stable fixed point. We compose with a
# nonlinear feedback to recover period doubling, mimicking time-delay coupling.
def bl_delay_map(a: float, T_star: float = 1.0, n_iter: int = 600,
                 n_keep: int = 200, T0: float = 0.5) -> np.ndarray:
    """Time-delay-style two-step map exhibiting period doubling.

    Composes two stages: poloidal P_{n+1} = a*T_n/(1+(T_n/T*)^2) and
    toroidal T_{n+1} = b*P_{n+1}*(1 - P_{n+1}) with b = 4 to allow folding.
    """
    T = T0
    series = np.empty(n_keep)
    for n in range(n_iter):
        P = a * T / (1.0 + (T / T_star) ** 2)
        T = 4.0 * P * (1.0 - P)
        T = max(min(T, 0.999), 1e-6)
        if n >= n_iter - n_keep:
            series[n - (n_iter - n_keep)] = T
    return series


a_values = np.linspace(0.5, 4.0, 600)
fig, ax = plt.subplots(figsize=(10, 5))
for a in a_values:
    attractor = bl_delay_map(a, T_star=1.0, n_iter=500, n_keep=120)
    ax.plot([a] * len(attractor), attractor, ",k", alpha=0.6)
ax.set_xlabel("a (linear B–L growth coefficient)")
ax.set_ylabel("T (toroidal amplitude attractor)")
ax.set_title("Bifurcation: period doubling → chaos / 분기: 주기배가 → 혼돈")
plt.tight_layout()
plt.show()

## Part 6 — Stochastic B–L Cycle: Grand Minima from Joy-Law Scatter / 확률적 B–L 주기: Joy 법칙 산포에서 나오는 Grand Minima

**English.** The B–L poloidal source is *intrinsically* stochastic because active-region tilt scatters by $\sim 30°$ around Joy's law. We model this by driving the cycle amplitude with multiplicative noise:
$$T_{n+1} = a\,T_n / [1 + (T_n/T_*)^2] \cdot (1 + \sigma\,\xi_n),$$
with $\xi_n$ standard normal. Long-tailed amplitude excursions reproduce Maunder-type quiescent epochs without invoking external forcing — Charbonneau §5.5.

**한국어.** B–L 폴로이달 원천은 활동영역 기울기가 Joy 법칙 주위로 약 $30°$ 산포하므로 *본질적으로* 확률적입니다. 곱셈 잡음으로 주기 진폭을 강제합니다. 긴 꼬리 진폭 변동이 외부 강제 없이 Maunder형 정적 시기를 재현합니다 — §5.5.

In [ ]:
def stochastic_bl(
    n_cycles: int = 800,
    a: float = 2.2,
    T_star: float = 1.0,
    sigma: float = 0.25,
    seed: int = 0,
) -> np.ndarray:
    """Stochastic B-L map with multiplicative Joy-law-scatter noise.

    Args:
        n_cycles: Number of cycles (each ~11 yr) to simulate.
        a: Linear B-L growth coefficient.
        T_star: Saturation amplitude.
        sigma: Noise amplitude (1-sigma fractional).
        seed: RNG seed.

    Returns:
        Array of cycle amplitudes T_n.
    """
    rng_local = np.random.default_rng(seed)
    T = np.empty(n_cycles)
    T[0] = 0.5
    for n in range(n_cycles - 1):
        det = a * T[n] / (1.0 + (T[n] / T_star) ** 2)
        T[n + 1] = max(0.0, det * (1.0 + sigma * rng_local.standard_normal()))
    return T


amps = stochastic_bl(n_cycles=600, a=2.2, sigma=0.30, seed=7)
cycle_idx = np.arange(amps.size)
year = cycle_idx * 11.0  # approximate 11-yr cycle

# Identify grand minima as runs where smoothed amplitude < 0.3 * median for >=3 cycles.
smoothed = np.convolve(amps, np.ones(3) / 3, mode="same")
threshold = 0.3 * np.median(amps)
is_min = smoothed < threshold

# Compute fraction of time spent in Maunder-like state via np.trapezoid.
frac_minimum = np.trapezoid(is_min.astype(float), year) / (year[-1] - year[0])

fig, ax = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
ax[0].plot(year, amps, "k-", lw=0.8)
ax[0].fill_between(year, 0, amps, where=is_min, color="orange", alpha=0.5,
                  label="grand minima episodes")
ax[0].axhline(threshold, color="r", ls="--", lw=0.8, label="minimum threshold")
ax[0].set_ylabel("cycle amplitude T_n")
ax[0].set_title(f"Stochastic B–L cycle, sigma=0.30; minimum fraction = {frac_minimum:.2%}")
ax[0].legend()

# Histogram of amplitudes — long-tailed.
ax[1].hist(amps, bins=50, color="steelblue", edgecolor="k", alpha=0.85)
ax[1].axvline(threshold, color="r", ls="--", label="minimum threshold")
ax[1].set_xlabel("cycle amplitude")
ax[1].set_ylabel("count")
ax[1].set_title("Amplitude distribution / 진폭 분포")
ax[1].legend()
plt.tight_layout()
plt.show()

print(f"Mean amplitude: {amps.mean():.3f}")
print(f"Std amplitude: {amps.std():.3f}")
print(f"Fraction of time in grand-minimum state: {frac_minimum:.3f}")

## Summary / 요약

**English.** This notebook implemented the central pedagogical models of Charbonneau (2010): (1) the helioseismic differential rotation profile that is the universal observational input; (2) the Parker dynamo wave demonstrating the $\alpha\Omega$ dispersion and migration; (3) a low-order coupled $\alpha\Omega$ system showing dynamo-number criticality; (4) a 1-D-in-latitude Babcock–Leighton flux-transport dynamo producing an equatorward butterfly and polar field reversal; (5) the time-delay iterative map exhibiting period doubling → chaos as a deterministic origin of cycle-amplitude variability; (6) a stochastic B–L cycle reproducing Maunder-like grand minima from Joy-law scatter alone.

**한국어.** 이 노트북은 Charbonneau (2010)의 핵심 교육적 모델을 구현했습니다: (1) 보편적 관측 입력인 일진동학 미분 회전, (2) $\alpha\Omega$ 분산과 이동을 보여주는 Parker 다이나모 파동, (3) 다이나모 수 임계성을 보여주는 저차원 결합 $\alpha\Omega$ 시스템, (4) 적도방향 버터플라이와 극성 역전을 만드는 위도 1차원 Babcock–Leighton 플럭스 수송 다이나모, (5) 결정론적 주기 진폭 변동성의 기원으로서 주기배가→혼돈을 보이는 시간지연 반복 사상, (6) Joy 법칙 산포만으로 Maunder형 grand minima를 재현하는 확률적 B–L 주기.